# 1. Save GraphCast activations to Google Drive

**Run this notebook first. No code changes are needed for your first run.**

1. Open this in **Google Colab in your browser**. Choose **Runtime → Change runtime type → A100 GPU** (or H100). Use a high-RAM runtime if available.
2. Choose **Runtime → Run all**, and approve the Google Drive connection when asked.
3. Wait for **“Finished”**, then open [notebook 2: Train TopK SAE](https://colab.research.google.com/github/observerhan/graphcast-interpretability/blob/kan-sae-training/02_train_topk_sae.ipynb).

The notebook runs the pretrained **0.25° GraphCast** and saves its **layer-8 activations** in `My Drive/AMS691_GraphCast/monthly_2017_2018/activations_layer8`.
Each snapshot contains 512 values for each of 40,962 locations. The **27 snapshots** use about **2.3 GB** in Drive and stay there after Colab disconnects.

**Hardware:** full GraphCast needs a large GPU; a standard free T4 is not sufficient for this workflow. The first forecast also compiles the model, so it is much slower than later forecasts. Setup and weather downloads may take several minutes each.

## Step 1 — Connect your Google Drive

Use the same Google account in both notebooks. The folder is created for you.

In [ ]:
# @title Connect Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_FOLDER = Path('/content/drive/MyDrive/AMS691_GraphCast/monthly_2017_2018')
ACTIVATION_DIR = PROJECT_FOLDER / 'activations_layer8'
ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)
print('Your project folder:', PROJECT_FOLDER)

## Step 2 — Review the training and validation dates

| Use | Weather dates at 12:00 UTC | Snapshots |
|---|---|---:|
| Train the SAE | The 15th of each month in 2017 and 2018 | 24 |
| Check reconstruction on other dates (validation) | August 29, 2017 · April 29, 2018 · December 29, 2018 | 3 |

We add one validation date for every ten training dates, rounding up. The dates are spread across the two years. **12:00 UTC** is the time in the weather record. You can run Colab at any time.

In [ ]:
# @title Dates to extract
from calendar import monthrange
from datetime import date
from math import ceil
import json

DAYS_OF_MONTH = [15]  # Optional: use [5, 15] for 48 training dates and 5 validation dates.
months = [(year, month) for year in (2017, 2018) for month in range(1, 13)]
TRAINING_DATES = [
    date(year, month, day).isoformat()
    for year, month in months
    for day in DAYS_OF_MONTH
]

# Space the validation dates over the two years. Use the 29th (28th in a short February).
validation_count = ceil(len(TRAINING_DATES) / 10)
VALIDATION_DATES = []
for group in range(1, validation_count + 1):
    month_index = ceil(group * len(months) / validation_count) - 1
    year, month = months[month_index]
    day = min(29, monthrange(year, month)[1])
    VALIDATION_DATES.append(date(year, month, day).isoformat())

DATES = sorted(TRAINING_DATES + VALIDATION_DATES)
if not TRAINING_DATES or len(DATES) != len(set(DATES)):
    raise ValueError('Use distinct training dates and separate validation dates.')

CONFIG_PATH = Path('/content/ams691_extraction_settings.json')
settings = {
    'dates': DATES,
    'validation_dates': VALIDATION_DATES,
    'activation_dir': str(ACTIVATION_DIR),
}
CONFIG_PATH.write_text(json.dumps(settings))
print(f'Training snapshots ({len(TRAINING_DATES)}):', ', '.join(TRAINING_DATES))
print(f'Validation snapshots ({len(VALIDATION_DATES)}):', ', '.join(VALIDATION_DATES))
print(f'Saving {len(DATES)} snapshots at 12:00 UTC to:', ACTIVATION_DIR)

## Step 3 — Set up GraphCast automatically

Run this cell unchanged. You do not need to install anything on your laptop or select package versions.

In [ ]:
# @title Load the project and prepare GraphCast
import os
import subprocess
import sys
from pathlib import Path

# One immutable code version is shared by all six notebooks.
RELEASE = 'b00fd40e75059f2ac345c70b273718a1cd945b94'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'setup_graphcast.py')], check=True)
PYTHON = '/content/ams691_graphcast_env/bin/python'


## Step 4 — Review the shared extraction program

The extraction functions and dataset address live in `scripts/ams691_extract.py` in the repository. Notebooks 03 and 05 use this same file. This cell shows the script path, checkpoint name, and ERA5 dataset address.


In [ ]:
# @title Show the extraction code and weather source
from ams691_extract import MODEL_NAME, WEATHER_STORE

EXTRACT_SCRIPT = REPO / 'scripts' / 'ams691_extract.py'
print('Extraction program:', EXTRACT_SCRIPT)
print('GraphCast checkpoint:', MODEL_NAME)
print('ERA5 dataset:', WEATHER_STORE)


## Step 5 — Run GraphCast and save your activations

This is the cell that **actually runs GraphCast**. Each completed snapshot is saved immediately to Drive.
If Colab disconnects, reopen this notebook and run it again; completed files will be skipped.

In [ ]:
# @title Extract and save to Drive
process = subprocess.Popen(
    [PYTHON, '-u', str(EXTRACT_SCRIPT), str(CONFIG_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ, 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'},
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
except BaseException:
    process.terminate()
    process.wait()
    raise
if returncode:
    raise RuntimeError('Extraction stopped. Completed snapshots are safe in Drive. Read the error above.')

manifest = json.loads((ACTIVATION_DIR / 'dataset.json').read_text())
assert set(manifest['requested_centers']) <= set(manifest['completed_centers'])
print('\nREADY FOR NOTEBOOK 2: your activations are saved in Google Drive.')

## Next

Choose **Runtime → Disconnect and delete runtime** to release the large GPU. Your Drive files will stay saved.
Then open [**2. Train TopK SAE**](https://colab.research.google.com/github/observerhan/graphcast-interpretability/blob/kan-sae-training/02_train_topk_sae.ipynb). A T4 GPU is sufficient for that notebook.

Based on the authors' [hurricane demo](https://github.com/theodoremacmillan/graphcast-interpretability/blob/main/hurricane_feature_demo.ipynb) and [GraphCast hooks](https://github.com/theodoremacmillan/graphcast/tree/sae-hooks). Public weather data are from [WeatherBench 2](https://weatherbench2.readthedocs.io/en/latest/data-guide.html).